In [1]:
# =========================================================
# Notebook 1 V6.0: Dataset Preparation (Face-Centric Ready)
# Purpose:
# This notebook prepares the dataset when each sample is already
# stored as a folder of extracted frames.
#
# Main V6 upgrades:
# - keeps the original frame-folder workflow
# - adds source-bias diagnostics (very important for deepfake data)
# - adds optional source-aware holdout evaluation split
# - saves richer CSV metadata used by Notebook 2 / Notebook 3
#
# NOTE:
# This notebook still works on the ORIGINAL frame folders.
# Face cropping is applied online in Notebook 2 / Notebook 3 so you do NOT
# need to regenerate the dataset manually right now.
# =========================================================


In [2]:
# Cell 1: Import required libraries
import os
import re
import glob
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# Cell 2: Paths and configuration
# ------------------------------------------------------------------
# IMPORTANT:
# This notebook is configured for datasets where:
#   each folder = one video sample
#   each folder contains extracted image frames
#
# Recommended structure for your current setup:
# processed/
#   Real/
#      old_real/
#         712/
#            frame_0001.jpg
#            frame_0002.jpg
#      new_real/
#         id25_0001/
#            0001.png
#            0002.png
#   Fake/
#      01_20_hugging_happy_FW94AIMJ/
#         frame_0001.jpg
#
# Keep REAL_ROOTS pointed to the parent Real directory so the notebook
# can infer:
#   old_real / new_real
#
# Keep FAKE_ROOTS pointed to the parent Fake directory. If fake samples
# are directly under Fake/, their source_name will become direct_fake.
# ------------------------------------------------------------------
BASE_DIR = r"D:\Courses\DEPI\project\DeepFake\processed"
SAVE_SPLIT_DIR = r"D:\Courses\DEPI\project\DeepFake\splits_v6"

REAL_ROOTS = [
    os.path.join(BASE_DIR, "Real"),
]

FAKE_ROOTS = [
    os.path.join(BASE_DIR, "Fake"),
]

MIN_FRAMES = 32

# Optional reference value used later by training/inference notebooks.
TARGET_SEQUENCE_LENGTH = 42

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

assert abs((TRAIN_SIZE + VAL_SIZE + TEST_SIZE) - 1.0) < 1e-8, "Split ratios must sum to 1."

os.makedirs(SAVE_SPLIT_DIR, exist_ok=True)

print("BASE_DIR               :", BASE_DIR)
print("SAVE_SPLIT_DIR         :", SAVE_SPLIT_DIR)
print("REAL_ROOTS             :", REAL_ROOTS)
print("FAKE_ROOTS             :", FAKE_ROOTS)
print("MIN_FRAMES             :", MIN_FRAMES)
print("TARGET_SEQUENCE_LENGTH :", TARGET_SEQUENCE_LENGTH)

# Optional advanced evaluation setting
# If True: one source from each class can be held out entirely from training
# and placed into test when possible. This helps expose source bias.
ENABLE_SOURCE_HOLDOUT_SPLIT = False

# Example format:
# SOURCE_HOLDOUT_MAP = {"Real": ["new_real"], "Fake": ["direct_fake"]}
# Leave empty to disable or if you do not yet have multiple fake sources.
SOURCE_HOLDOUT_MAP = {
    "Real": [],
    "Fake": []
}

print("ENABLE_SOURCE_HOLDOUT_SPLIT:", ENABLE_SOURCE_HOLDOUT_SPLIT)


BASE_DIR               : D:\Courses\DEPI\project\DeepFake\processed
SAVE_SPLIT_DIR         : D:\Courses\DEPI\project\DeepFake\splits_v6
REAL_ROOTS             : ['D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real']
FAKE_ROOTS             : ['D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Fake']
MIN_FRAMES             : 32
TARGET_SEQUENCE_LENGTH : 42
ENABLE_SOURCE_HOLDOUT_SPLIT: False


In [4]:
# Cell 3: Helper functions
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def natural_key(text):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(text))]

def get_frame_paths(folder_path: str):
    valid_exts = {ext.lower() for ext in IMAGE_EXTS}
    frame_paths = []
    for name in os.listdir(folder_path):
        full_path = os.path.join(folder_path, name)
        if os.path.isfile(full_path):
            ext = os.path.splitext(name)[1].lower()
            if ext in valid_exts:
                frame_paths.append(full_path)
    return sorted(frame_paths, key=natural_key)

def folder_contains_frames(folder_path: str):
    return len(get_frame_paths(folder_path)) > 0

def discover_video_folders(root_dir: str):
    """
    Recursively scan a class/source root and return only folders that
    directly contain image frames. Each discovered folder is treated
    as ONE video sample.
    """
    discovered = []
    for current_root, dirs, files in os.walk(root_dir):
        image_files = [f for f in files if f.lower().endswith(IMAGE_EXTS)]
        if image_files:
            discovered.append(current_root)
    return sorted(discovered, key=natural_key)

def infer_source_name(root_dir: str, folder_path: str):
    """
    Infer source name relative to the class root.

    Examples:
        root_dir = processed/Real
        folder   = processed/Real/old_real/712      -> source = old_real
        folder   = processed/Real/new_real/id25_01  -> source = new_real
        folder   = processed/Real/712               -> source = direct_real

        root_dir = processed/Fake
        folder   = processed/Fake/fake_001          -> source = direct_fake
        folder   = processed/Fake/source_a/fake_7   -> source = source_a
    """
    rel = os.path.relpath(folder_path, root_dir)
    parts = Path(rel).parts
    class_root_name = Path(root_dir).name.lower()

    # If the sample folder is directly under the class root:
    # Real/<video_folder> -> direct_real
    # Fake/<video_folder> -> direct_fake
    if len(parts) <= 1:
        return f"direct_{class_root_name}"

    # Otherwise use the first subfolder under the class root.
    return parts[0]

def build_video_id(root_dir: str, folder_path: str):
    """
    Build a unique video_id from the relative path.
    """
    rel = os.path.relpath(folder_path, root_dir)
    rel_no_ext = str(Path(rel).with_suffix(""))
    return rel_no_ext.replace("\\", "__").replace("/", "__")

def uniform_sample_paths(frame_paths, target_len=32):
    """
    Return uniformly sampled frame paths for quick inspection or later use.
    This does NOT modify saved images on disk.
    """
    if len(frame_paths) == 0:
        return []
    if len(frame_paths) <= target_len:
        return frame_paths
    indices = np.linspace(0, len(frame_paths) - 1, target_len, dtype=int)
    return [frame_paths[i] for i in indices]

def collect_samples(root_dirs, label_value: int, class_name: str, min_frames: int = 32, target_len: int = 32):
    if isinstance(root_dirs, str):
        root_dirs = [root_dirs]

    valid_samples = []
    invalid_samples = []

    for root_dir in root_dirs:
        video_folders = discover_video_folders(root_dir)

        for folder_path in video_folders:
            frame_paths = get_frame_paths(folder_path)
            n_frames = len(frame_paths)
            source_name = infer_source_name(root_dir, folder_path)
            sampled_preview = uniform_sample_paths(frame_paths, target_len=target_len)

            record = {
                "video_id": build_video_id(root_dir, folder_path),
                "folder_name": os.path.basename(folder_path),
                "folder_path": folder_path,
                "label": label_value,
                "class_name": class_name,
                "source_name": source_name,
                "num_frames": n_frames,
                "first_frame": frame_paths[0] if n_frames > 0 else None,
                "last_frame": frame_paths[-1] if n_frames > 0 else None,
                "sampled_frame_count": len(sampled_preview),
                "dataset_split": None,
            }

            if n_frames >= min_frames:
                valid_samples.append(record)
            else:
                invalid_samples.append(record)

    return valid_samples, invalid_samples


In [5]:
# Cell 4: Collect Real and Fake frame-folder samples
real_samples, real_invalid = collect_samples(
    REAL_ROOTS,
    label_value=1,
    class_name="Real",
    min_frames=MIN_FRAMES,
    target_len=TARGET_SEQUENCE_LENGTH
)

fake_samples, fake_invalid = collect_samples(
    FAKE_ROOTS,
    label_value=0,
    class_name="Fake",
    min_frames=MIN_FRAMES,
    target_len=TARGET_SEQUENCE_LENGTH
)

all_samples = real_samples + fake_samples
invalid_samples = real_invalid + fake_invalid

data_df = pd.DataFrame(all_samples)
invalid_df = pd.DataFrame(invalid_samples)

if len(data_df) == 0:
    raise ValueError("No valid frame-folder samples were found. Check your Real/Fake roots.")

# Remove any accidental duplicate folder_path entries
before_dedup = len(data_df)
data_df = data_df.drop_duplicates(subset=["folder_path"]).reset_index(drop=True)
after_dedup = len(data_df)

# Shuffle after deduplication
data_df = data_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Valid frame-folder samples:", len(data_df))
print("Removed duplicate folder entries:", before_dedup - after_dedup)
print("\nClass counts:")
print(data_df["class_name"].value_counts())

print("\nSource counts:")
display(
    data_df.groupby(["class_name", "source_name"])
           .size()
           .reset_index(name="count")
           .sort_values(["class_name", "count"], ascending=[True, False])
)

print(f"\nInvalid samples (< {MIN_FRAMES} frames):", len(invalid_df))
display(data_df.head())


Valid frame-folder samples: 3471
Removed duplicate folder entries: 0

Class counts:
class_name
Fake    1888
Real    1583
Name: count, dtype: int64

Source counts:


,class_name,source_name,count
2,Fake,old_fake,698
0,Fake,new_fake,637
1,Fake,old2_fake,553
3,Real,new_real,587
4,Real,old2_real,498
5,Real,old_real,498



Invalid samples (< 32 frames): 118


,video_id,folder_name,folder_path,label,class_name,source_name,num_frames,first_frame,last_frame,sampled_frame_count,dataset_split
0,new_fake__01_20__hugging_happy__FW94AIMJ,01_20__hugging_happy__FW94AIMJ,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,100,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,42,None
1,old_fake__450_533,450_533,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,old_fake,100,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,42,None
2,new_fake__11_06__outside_talking_pan_laughing_...,11_06__outside_talking_pan_laughing__C8C4FW99,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,53,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,42,None
3,new_real__id34_0005,id34_0005,D:\Courses\DEPI\project\DeepFake\processed\Rea...,1,Real,new_real,100,D:\Courses\DEPI\project\DeepFake\processed\Rea...,D:\Courses\DEPI\project\DeepFake\processed\Rea...,42,None
4,old2_real__900,900,D:\Courses\DEPI\project\DeepFake\processed\Rea...,1,Real,old2_real,100,D:\Courses\DEPI\project\DeepFake\processed\Rea...,D:\Courses\DEPI\project\DeepFake\processed\Rea...,42,None


In [6]:
# Cell 5: Dataset quality checks
print("Frame statistics:")
display(data_df["num_frames"].describe())

print("Smallest frame-count samples:")
display(data_df.sort_values("num_frames").head(10))

print("How many samples are below common thresholds?")
for thr in [32, 40, 48, 64, 80, 100]:
    count = int((data_df["num_frames"] < thr).sum())
    print(f"  < {thr:>3}: {count}")

dup_video_ids = data_df["video_id"].duplicated().sum()
print("\nDuplicate video_id count:", dup_video_ids)

print("\nPer-source summary:")
display(
    data_df.groupby(["class_name", "source_name"])
           .agg(samples=("video_id", "count"),
                min_frames=("num_frames", "min"),
                median_frames=("num_frames", "median"),
                max_frames=("num_frames", "max"))
           .reset_index()
           .sort_values(["class_name", "samples"], ascending=[True, False])
)

Frame statistics:


count    3471.000000
mean       97.457505
std        10.044877
min        32.000000
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: num_frames, dtype: float64

Smallest frame-count samples:


,video_id,folder_name,folder_path,label,class_name,source_name,num_frames,first_frame,last_frame,sampled_frame_count,dataset_split
2429,new_fake__08_05__walk_down_hall_angry__FBICSP2C,08_05__walk_down_hall_angry__FBICSP2C,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,32,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,32,None
1597,new_fake__04_27__walking_outside_cafe_disguste...,04_27__walking_outside_cafe_disgusted__2CCI2ND1,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,32,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,32,None
1189,old2_fake__20_21__secret_conversation__ZCB5OMEW,20_21__secret_conversation__ZCB5OMEW,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,old2_fake,32,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,32,None
1213,new_fake__02_13__secret_conversation__CP5HFV3K,02_13__secret_conversation__CP5HFV3K,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,32,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,32,None
1231,old_fake__427_637,427_637,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,old_fake,34,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,34,None
55,old2_fake__25_27__walk_down_hall_angry__1KU29EUL,25_27__walk_down_hall_angry__1KU29EUL,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,old2_fake,34,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,34,None
986,new_fake__15_02__walking_outside_cafe_disguste...,15_02__walking_outside_cafe_disgusted__P2FAY3DR,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,35,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,35,None
1361,new_fake__02_09__exit_phone_room__HIH8YA82,02_09__exit_phone_room__HIH8YA82,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,35,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,35,None
2152,old2_fake__28_16__walking_down_street_outside_...,28_16__walking_down_street_outside_angry__6DWL...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,old2_fake,35,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,35,None
1107,new_fake__13_03__outside_talking_still_laughin...,13_03__outside_talking_still_laughing__GBYWJW06,D:\Courses\DEPI\project\DeepFake\processed\Fak...,0,Fake,new_fake,35,D:\Courses\DEPI\project\DeepFake\processed\Fak...,D:\Courses\DEPI\project\DeepFake\processed\Fak...,35,None


How many samples are below common thresholds?
  <  32: 0
  <  40: 21
  <  48: 41
  <  64: 118
  <  80: 179
  < 100: 314

Duplicate video_id count: 0

Per-source summary:


,class_name,source_name,samples,min_frames,median_frames,max_frames
2,Fake,old_fake,698,34,100.0,100
0,Fake,new_fake,637,32,100.0,100
1,Fake,old2_fake,553,32,100.0,100
3,Real,new_real,587,70,100.0,100
4,Real,old2_real,498,95,100.0,100
5,Real,old_real,498,36,100.0,100


In [7]:
# Cell 6: Train / Val / Test split
# ------------------------------------------------------------------
# We prefer stratifying by both label and source when possible.
# This is useful when you add new Real sources, so the split does not
# accidentally place one source only in train or only in test.
# If the combined stratification becomes too sparse, the notebook
# falls back safely to label-only stratification.
# ------------------------------------------------------------------
def choose_stratify_column(df: pd.DataFrame):
    combined = df["class_name"].astype(str) + "__" + df["source_name"].astype(str)
    value_counts = combined.value_counts()

    # Combined stratification is only safe if each group has at least 2 samples
    if len(value_counts) > 1 and value_counts.min() >= 2:
        print("Using combined stratification: class_name + source_name")
        return combined

    print("Falling back to label-only stratification")
    return df["label"]

stratify_main = choose_stratify_column(data_df)

train_df, temp_df = train_test_split(
    data_df,
    test_size=(VAL_SIZE + TEST_SIZE),
    stratify=stratify_main,
    random_state=SEED
)

relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
stratify_temp = choose_stratify_column(temp_df)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    stratify=stratify_temp,
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df["dataset_split"] = "train"
val_df["dataset_split"] = "val"
test_df["dataset_split"] = "test"

print("Train size:", len(train_df))
print(train_df["class_name"].value_counts(), "\n")

print("Val size:", len(val_df))
print(val_df["class_name"].value_counts(), "\n")

print("Test size:", len(test_df))
print(test_df["class_name"].value_counts())

print("\nSplit distribution by source:")
split_overview = pd.concat([train_df, val_df, test_df], axis=0)
display(
    split_overview.groupby(["dataset_split", "class_name", "source_name"])
                  .size()
                  .reset_index(name="count")
                  .sort_values(["dataset_split", "class_name", "count"], ascending=[True, True, False])
)

Using combined stratification: class_name + source_name
Using combined stratification: class_name + source_name
Train size: 2429
class_name
Fake    1321
Real    1108
Name: count, dtype: int64 

Val size: 521
class_name
Fake    283
Real    238
Name: count, dtype: int64 

Test size: 521
class_name
Fake    284
Real    237
Name: count, dtype: int64

Split distribution by source:


,dataset_split,class_name,source_name,count
2,test,Fake,old_fake,105
0,test,Fake,new_fake,96
1,test,Fake,old2_fake,83
3,test,Real,new_real,88
4,test,Real,old2_real,75
5,test,Real,old_real,74
8,train,Fake,old_fake,488
6,train,Fake,new_fake,446
7,train,Fake,old2_fake,387
9,train,Real,new_real,411


In [8]:
# Cell 7: Save split CSV files and summary
train_csv = os.path.join(SAVE_SPLIT_DIR, "train_df.csv")
val_csv = os.path.join(SAVE_SPLIT_DIR, "val_df.csv")
test_csv = os.path.join(SAVE_SPLIT_DIR, "test_df.csv")
invalid_csv = os.path.join(SAVE_SPLIT_DIR, "invalid_samples.csv")
summary_json = os.path.join(SAVE_SPLIT_DIR, "split_summary_v6.json")

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

if len(invalid_df) > 0:
    invalid_df.to_csv(invalid_csv, index=False)

summary = {
    "seed": SEED,
    "min_frames": MIN_FRAMES,
    "target_sequence_length": TARGET_SEQUENCE_LENGTH,
    "real_roots": REAL_ROOTS,
    "fake_roots": FAKE_ROOTS,
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "train_label_counts": train_df["label"].value_counts().to_dict(),
    "val_label_counts": val_df["label"].value_counts().to_dict(),
    "test_label_counts": test_df["label"].value_counts().to_dict(),
    "overall_num_frames_stats": data_df["num_frames"].describe().to_dict(),
    "source_distribution": (
        data_df.groupby(["class_name", "source_name"]).size().reset_index(name="count").to_dict(orient="records")
    ),
    "invalid_samples_count": int(len(invalid_df))
}

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Splits saved successfully.")
print("Saved files:")
print(" -", train_csv)
print(" -", val_csv)
print(" -", test_csv)
if len(invalid_df) > 0:
    print(" -", invalid_csv)
print(" -", summary_json)


Splits saved successfully.
Saved files:
 - D:\Courses\DEPI\project\DeepFake\splits_v6\train_df.csv
 - D:\Courses\DEPI\project\DeepFake\splits_v6\val_df.csv
 - D:\Courses\DEPI\project\DeepFake\splits_v6\test_df.csv
 - D:\Courses\DEPI\project\DeepFake\splits_v6\invalid_samples.csv
 - D:\Courses\DEPI\project\DeepFake\splits_v6\split_summary_v6.json


In [9]:
# Cell 8: Quick manual inspection from one frame folder
sample_row = train_df.iloc[0]
sample_folder = sample_row["folder_path"]
sample_frames = get_frame_paths(sample_folder)
sampled_preview = uniform_sample_paths(sample_frames, target_len=TARGET_SEQUENCE_LENGTH)

print("Sample video_id         :", sample_row["video_id"])
print("Sample folder_name      :", sample_row["folder_name"])
print("Sample source           :", sample_row["source_name"])
print("Sample class            :", sample_row["class_name"])
print("Sample label            :", sample_row["label"])
print("Original frames in folder:", len(sample_frames))
print("Uniform sampled preview :", len(sampled_preview))
print("First 5 original frames:")
print(sample_frames[:5])
print("\nFirst 5 sampled frames:")
print(sampled_preview[:5])


Sample video_id         : new_real__id58_0002
Sample folder_name      : id58_0002
Sample source           : new_real
Sample class            : Real
Sample label            : 1
Original frames in folder: 100
Uniform sampled preview : 42
First 5 original frames:
['D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_000.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_001.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_002.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_003.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_004.jpg']

First 5 sampled frames:
['D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_000.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_real\\id58_0002\\frame_002.jpg', 'D:\\Courses\\DEPI\\project\\DeepFake\\processed\\Real\\new_r

In [10]:
print("Train distribution by class + source:")
print(train_df.groupby(["class_name", "source_name"]).size(), "\n")

print("Val distribution by class + source:")
print(val_df.groupby(["class_name", "source_name"]).size(), "\n")

print("Test distribution by class + source:")
print(test_df.groupby(["class_name", "source_name"]).size(), "\n")

Train distribution by class + source:
class_name  source_name
Fake        new_fake       446
            old2_fake      387
            old_fake       488
Real        new_real       411
            old2_real      348
            old_real       349
dtype: int64 

Val distribution by class + source:
class_name  source_name
Fake        new_fake        95
            old2_fake       83
            old_fake       105
Real        new_real        88
            old2_real       75
            old_real        75
dtype: int64 

Test distribution by class + source:
class_name  source_name
Fake        new_fake        96
            old2_fake       83
            old_fake       105
Real        new_real        88
            old2_real       75
            old_real        74
dtype: int64 



In [11]:
# Cell 8.5: Source-bias diagnostics (VERY IMPORTANT for deepfake projects)
print("\nSource-bias diagnostic summary:")
source_table = (
    data_df.groupby(["class_name", "source_name"])
          .size()
          .reset_index(name="count")
          .sort_values(["class_name", "count"], ascending=[True, False])
)
display(source_table)

real_sources = sorted(data_df.loc[data_df["class_name"] == "Real", "source_name"].unique().tolist())
fake_sources = sorted(data_df.loc[data_df["class_name"] == "Fake", "source_name"].unique().tolist())

print("Real sources :", real_sources)
print("Fake sources :", fake_sources)

if len(fake_sources) <= 1:
    print("⚠️ Warning: Fake class currently has only ONE source. This can cause strong source bias.")
if len(real_sources) <= 1:
    print("⚠️ Warning: Real class currently has only ONE source.")

print("\nRecommendation:")
print("- Add more fake sources and more real sources before expecting a big accuracy jump.")
print("- Keep preprocessing consistent across all sources.")



Source-bias diagnostic summary:


,class_name,source_name,count
2,Fake,old_fake,698
0,Fake,new_fake,637
1,Fake,old2_fake,553
3,Real,new_real,587
4,Real,old2_real,498
5,Real,old_real,498


Real sources : ['new_real', 'old2_real', 'old_real']
Fake sources : ['new_fake', 'old2_fake', 'old_fake']

Recommendation:
- Add more fake sources and more real sources before expecting a big accuracy jump.
- Keep preprocessing consistent across all sources.
